# Proyecto Integrador (TC5035)
# Identificación de Gestos en el Lenguaje de Manos Mexicano
# Avance 4. Modelos alternativos
---
## EQUIPO 51
- CARLOS MIGUEL ARVIZU DURÁN - A01410682
- YOHANNA CEBALLOS SALOMÓN - A01795115
- RUBÉN DÍAZ GARCÍA - A01371849

## Profesores
- PROFESOR ASESOR: DR. RAÚL VALENTE RAMÍREZ VELARDE
- PROFESORA TITULAR: DRA. GRETTEL BARCELÓ ALONSO
- PROFESOR TITULAR: DR. LUIS EDUARDO FALCÓN MORALES
- PROFESORA ASISTENTE: MTRA. VERÓNICA SANDRA GUZMÁN DE VALLE

---

# Modelo 5 Gradient Boosting (XGBoost)

## Librerías

In [1]:
import os
import cv2
import joblib
import json
import mediapipe as mp
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import xgboost as xgb
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm

OpenCV bindings requires "numpy" package.
Install it via command:
    pip install numpy


ModuleNotFoundError: No module named 'numpy.core'

## Carga de Datos

In [ ]:
# CONFIGURACIÓN: se debe ajustar a la carpeta desde donde se corre el programa
notebook_dir = Path.cwd() 
current_script_dir = notebook_dir#.parent
parent_dir = current_script_dir
PROCESSED_DATA_DIR = parent_dir / "processed_data_pipeline"
data_dir = parent_dir / "data"
VIDEOS_DIR =  data_dir / "videos"

print(f"\nPaths: \nPROCESSED DATA DIRECTORY: {PROCESSED_DATA_DIR} \nVIDEOS DIRECTORY: {VIDEOS_DIR}")

In [ ]:
# --- Cargar los Datos Preprocesados ---
print("--- Cargando datos preprocesados ---")
X_train = np.load(os.path.join(PROCESSED_DATA_DIR, "X_train.npy"))
X_test = np.load(os.path.join(PROCESSED_DATA_DIR, "X_test.npy"))
y_train_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "y_train.csv"))
y_test_df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, "y_test.csv"))
y_train_labels = y_train_df['class'].values
y_test_labels = y_test_df['class'].values

# Esta función convierte una secuencia (150, 5) en un vector de características plano
def create_features(sequence_data):
    num_samples = sequence_data.shape[0]
    # Calcularemos 6 estadísticas para cada una de las 5 mediciones = 30 características
    features = np.zeros((num_samples, 5 * 6))
    
    for i in range(num_samples):
        sample = sequence_data[i]
        # Para cada una de las 5 mediciones...
        feature_vector = []
        for j in range(5):
            measurement = sample[:, j]
            feature_vector.append(np.mean(measurement))
            feature_vector.append(np.std(measurement))
            feature_vector.append(np.min(measurement))
            feature_vector.append(np.max(measurement))
            feature_vector.append(np.median(measurement))
            feature_vector.append(np.var(measurement))
        features[i] = np.array(feature_vector)
        
    return features

X_train_features = create_features(X_train)
X_test_features = create_features(X_test)

print(f"X_train forma: {X_train_features.shape}")
print(f"X_test forma: {X_test_features.shape}")

# --- Codificar las Etiquetas ---
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train_labels)
y_test_encoded = label_encoder.transform(y_test_labels)

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_features,
    y_train_encoded,
    test_size=0.2, # Usamos el 20% de los datos de entrenamiento para validar
    random_state=42,
    stratify=y_train_encoded # Mantiene la proporción de clases
)


## Modelo 5 Gradient Boosting (XGBoost)

In [ ]:
# --- Construir el Modelo ---
print("\n--- Construyendo la arquitectura del modelo XGBoost ---")
model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    use_label_encoder=False,
    eval_metric='mlogloss'
)

## Entrenamiento

In [ ]:
model.fit(
    X_train_split,
    y_train_split,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=15, # Detiene el entrenamiento si el error en el set de validación no mejora en 15 rondas.
    verbose=True
)

## Resultado

In [ ]:
# --- Evaluar y Visualizar los Resultados ---
print("\n--- Evaluando el modelo con métricas detalladas ---")

# Realizar predicciones en el conjunto de prueba
y_pred = model_xgb.predict(X_test_features)
report = classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_)
print(report)

In [ ]:
# Generar y mostrar el Reporte de Clasificación
print("\n--- Reporte de Clasificación ---")
# Reporte: precisión, recall y f1-score para cada clase.
report = classification_report(y_test_encoded, y_pred, target_names=label_encoder.classes_)
print(report)

# Accuracy (Precisión Global)
accuracy = accuracy_score(y_test_encoded, y_pred)
print(f"La precisión (accuracy) del modelo es: {accuracy:.2f}")

# Recall
recall = recall_score(y_test_encoded, y_pred, average='weighted')
print(f"El recall (weighted) del modelo es: {recall:.2f}")

# F1-score
f1 = f1_score(y_test_encoded, y_pred, average='weighted')
print(f"El F1-score (weighted) del modelo es: {f1:.2f}")

In [ ]:
# Generar y mostrar la Matriz de Confusión
print("\n--- Matriz de Confusión ---")
cm = confusion_matrix(y_test_encoded, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title('Matriz de Confusión XGBoost', fontsize=16)
plt.ylabel('Clase Real', fontsize=12)
plt.xlabel('Clase Predicha', fontsize=12)
plt.show()

In [ ]:
# Graficar la precisión y la pérdida durante el entrenamiento
results = model.evals_result()
epochs = len(results['validation_0']['mlogloss'])
x_axis = range(0, epochs)

# 2. Graficar los resultados
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_axis, results['validation_0']['mlogloss'], label='Entrenamiento (Train)')
ax.plot(x_axis, results['validation_1']['mlogloss'], label='Validación (Val)')
ax.legend()
plt.ylabel('Pérdida (Log Loss)')
plt.xlabel('Ronda de Boosting')
plt.title('Curva de Pérdida del Entrenamiento de XGBoost')
plt.grid(True)
plt.show()

# print("\n--- Gráfica de precisión y pérdida durante el entrenamiento ---")
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
# # Gráfica de Precisión
# ax1.plot(history.history['accuracy'], label='Precisión de Entrenamiento')
# ax1.plot(history.history['val_accuracy'], label='Precisión de Validación')
# ax1.set_title('Precisión del Modelo')
# ax1.set_xlabel('Época')
# ax1.set_ylabel('Precisión')
# ax1.legend()
# ax1.grid(True)
# # Gráfica de Pérdida
# ax2.plot(history.history['loss'], label='Pérdida de Entrenamiento')
# ax2.plot(history.history['val_loss'], label='Pérdida de Validación')
# ax2.set_title('Pérdida del Modelo')
# ax2.set_xlabel('Época')
# ax2.set_ylabel('Pérdida')
# ax2.legend()
# ax2.grid(True)
# plt.suptitle('Historial de Entrenamiento del Modelo XGBoost', fontsize=16)
# plt.show()

print("\nEntrenamiento completado!")